# 3. Classification — Predicting `target_class`
**APRAU 2026/27 — Group 18**

Methods: Logistic Regression and Linear Discriminant Analysis (LDA)

Resampling: Holdout | CV k=5 | CV k=10 | LOOCV | Bootstrap

## 1. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = '/content/drive/MyDrive/APRAU/dataset_grupo_18.csv'
else:
    DATASET_PATH = '../dataset_grupo_18.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    LeaveOneOut, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import resample

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(DATASET_PATH)

df_enc = df.copy()
le = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

TARGET = 'target_class'
FEATURES = [c for c in df_enc.columns if c not in [TARGET, 'target_regression']]

X = df_enc[FEATURES].values
y = df_enc[TARGET].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'X: {X_scaled.shape} | Classes: {np.unique(y)}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LDA': LinearDiscriminantAnalysis()
}

results = []

def record(model_name, method, scores):
    results.append({
        'Model': model_name,
        'Method': method,
        'Mean Accuracy': np.mean(scores),
        'Std': np.std(scores)
    })

## 3. Holdout

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    record(name, 'Holdout', [acc])
    print(f'{name} — Holdout Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred))

    disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred))
    disp.plot()
    plt.title(f'{name} — Confusion Matrix (Holdout)')
    plt.show()

## 4. Cross-Validation (k=5 and k=10)

In [ ]:
for k in [5, 10]:
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    for name, model in models.items():
        scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
        record(name, f'CV k={k}', scores)
        print(f'{name} — CV k={k}: {np.mean(scores):.4f} ± {np.std(scores):.4f}')

## 5. Leave-One-Out Cross-Validation (LOOCV)

In [ ]:
# LOOCV is slow for large datasets — consider subsampling if needed
loo = LeaveOneOut()
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=loo, scoring='accuracy')
    record(name, 'LOOCV', scores)
    print(f'{name} — LOOCV: {np.mean(scores):.4f} ± {np.std(scores):.4f}')

## 6. Bootstrap

In [ ]:
N_BOOTSTRAP = 100

for name, model in models.items():
    boot_scores = []
    for i in range(N_BOOTSTRAP):
        X_boot, y_boot = resample(X_scaled, y, random_state=i)
        # OOB indices
        oob_mask = np.ones(len(X_scaled), dtype=bool)
        boot_idx = np.unique(resample(np.arange(len(X_scaled)), random_state=i))
        oob_mask[boot_idx] = False
        if oob_mask.sum() == 0:
            continue
        model.fit(X_boot, y_boot)
        acc = accuracy_score(y[oob_mask], model.predict(X_scaled[oob_mask]))
        boot_scores.append(acc)
    record(name, 'Bootstrap', boot_scores)
    print(f'{name} — Bootstrap: {np.mean(boot_scores):.4f} ± {np.std(boot_scores):.4f}')

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(['Model', 'Mean Accuracy'], ascending=[True, False])
results_df

In [ ]:
pivot = results_df.pivot(index='Method', columns='Model', values='Mean Accuracy')
pivot.plot(kind='bar', figsize=(12, 5))
plt.title('Accuracy by Model and Resampling Method')
plt.ylabel('Mean Accuracy')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Discussion

*(Fill in after running the models)*

- **Best method/model combination:**
- **How variance changes across resampling methods:**
- **LR vs LDA — which is more appropriate and why:**